In [1]:
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 4.1 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 4.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 28.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 57.8 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 88.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 93.2 MB/s  0:00:33m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 87.5 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 98.5 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 84.1 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 86.0 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 95.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 92.7 MB/s  0:00:0

In [10]:
# -*- coding: utf-8 -*-
"""
Pipeline complet de classification des datasets data.gouv.fr.

Chaîne de traitement en une seule exécution :
  1. Lit la taxonomie `type-donnees-mobi-mc.csv` (thèmes / mots-clés thème /
     sous-thèmes / termes).
  2. Charge le catalogue `datasets_mobi.csv` une seule fois.
  3. Pour chaque thème : filtre les datasets du thème (mots-clés du thème),
     puis calcule les scores (cosinus) et marges vers CHAQUE sous-thème
     (y compris les sous-thèmes des autres thèmes comme concurrents ? non :
     les concurrents sont les sous-thèmes du même thème, cf. remarque).
  4. Écrit les sorties :
     - datasets__<theme>.csv           : datasets du thème + colonnes
                                         score__/marge__ pour chaque sous-thème
                                         DU THÈME (validation manuelle)
     - par_sous_theme__<theme>__<st>.csv : datasets retenus du sous-thème
                                         (marge >= seuil), triés par marge
     - synthese.csv                     : une ligne par (thème, sous-thème)
                                         avec effectifs et stats de marge

Les fonctions de filtrage et de scoring sont reprises des scripts précédents
(filtre thématique, scoring embeddings) et légèrement adaptées pour le
traitement en lot : le catalogue et le modèle ne sont chargés qu'une fois.
"""

import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# --------------------------------------------------------------------------- #
# Paramètres
# --------------------------------------------------------------------------- #

CHEMIN_SOUS_THEMES = "data/type-donnees-mobi-mc.csv"
# CHEMIN_CATALOGUE = "data/datasets_mobi.csv"  sélection d'env 23k
CHEMIN_CATALOGUE = "data/datasets.csv" # tous les 75k datasets de data.gouv.fr
DOSSIER_SORTIE = Path("sorties_classification")   # créé s'il n'existe pas

MODELE = "intfloat/multilingual-e5-large"    # ou "intfloat/multilingual-e5-base"
SEUIL_MARGE = 0.03          # marge minimale vs meilleur sous-thème concurrent
TAILLE_LOT = 64             # taille de lot pour les embeddings (mémoire CPU)
MAX_CARACT_DESCRIPTION = 1500

COL_MOTS_CLES_THEME = "Mots-clés thème"
COLONNES_TEXTE = ["title", "tags", "description"]

PREFIXE_QUERY = "query: "
PREFIXE_PASSAGE = "passage: "




In [3]:
# --------------------------------------------------------------------------- #
# Utilitaires texte
# --------------------------------------------------------------------------- #

def _normaliser(texte) -> str:
    """Minuscules + suppression des accents."""
    if texte is None or (isinstance(texte, float) and pd.isna(texte)):
        return ""
    texte = str(texte).lower()
    texte = unicodedata.normalize("NFKD", str(texte))
    return "".join(c for c in texte if not unicodedata.combining(c))


def _nettoyer(texte) -> str:
    """Espaces multiples -> espace unique (pour la concaténation de champs)."""
    if texte is None or (isinstance(texte, float) and pd.isna(texte)):
        return ""
    return re.sub(r"\s+", " ", str(texte)).strip()


def texte_dataset(row: pd.Series) -> str:
    """Texte représentant un dataset : tags et titre surpondérés + description tronquée."""
    tags = _nettoyer(row.get("tags")).replace(",", " ")
    titre = _nettoyer(row.get("title"))
    description = _nettoyer(row.get("description"))[:MAX_CARACT_DESCRIPTION]
    return f"{tags} {tags} {titre} {titre} {description}"


def phrases_sous_theme(termes: str) -> list[str]:
    """Termes descriptifs d'un sous-thème découpés en phrases individuelles."""
    return [t.strip() for t in str(termes).split(",") if t.strip()]


In [4]:

# --------------------------------------------------------------------------- #
# Étape 1 : taxonomie
# --------------------------------------------------------------------------- #

def charger_taxonomie(chemin=CHEMIN_SOUS_THEMES) -> pd.DataFrame:
    """
    Charge la taxonomie. Formats acceptés :
    - nouveau : colonne "Mots-clés thème" séparée du nom du thème
    - ancien   : la colonne "Thème" liste nom + termes séparés par des virgules
    """
    taxo = pd.read_csv(chemin, sep=",", dtype=str)
    taxo.columns = [c.strip().strip("\ufeff") for c in taxo.columns]
    renommages = {c: COL_MOTS_CLES_THEME for c in taxo.columns
                  if _normaliser(c) in ("mots-cles theme", "mots-cles-theme")}
    taxo = taxo.rename(columns=renommages)
    return taxo.fillna("")


def themes_de_la_taxonomie(taxo: pd.DataFrame) -> dict[str, list[str]]:
    """
    Renvoie {nom de thème: [termes / mots-clés du thème]}.

    Au nouveau format, chaque ligne porte un nom de thème et une cellule
    "Mots-clés thème" ; on agrège par nom de thème (plusieurs sous-thèmes
    peuvent partager le même thème avec des mots-clés identiques -> union).
    À l'ancien format, le premier terme de la cellule "Thème" est considéré
    comme le nom du thème, les suivants comme des variantes de recherche.
    """
    themes: dict[str, dict[str, None]] = {}
    for _, row in taxo.iterrows():
        cell = row["Thème"]
        parties = [p.strip() for p in str(cell).split(",") if p.strip()]
        if not parties:
            continue
        nom_theme = parties[0]
        variantes = parties if COL_MOTS_CLES_THEME not in taxo.columns else parties
        termes = variantes
        if COL_MOTS_CLES_THEME in taxo.columns:
            termes = [t.strip() for t in str(row[COL_MOTS_CLES_THEME]).split(",")
                      if t.strip()]
        themes.setdefault(nom_theme, {})
        for t in termes:
            themes[nom_theme][t] = None
    return {nom: list(mots) for nom, mots in themes.items()}


def sous_themes_du_theme(taxo: pd.DataFrame, nom_theme: str) -> dict[str, str]:
    """Dict {sous-thème: termes} pour un thème (nouveau ou ancien format)."""
    lignes = taxo[taxo["Thème"].apply(
        lambda cell: _normaliser(nom_theme)
        in [_normaliser(p) for p in str(cell).split(",")]
    )]
    return {
        row["Sous-thème"].strip(): row["Termes"].strip()
        for _, row in lignes.iterrows()
    }



In [5]:
# --------------------------------------------------------------------------- #
# Étape 2 : filtrage du catalogue par thème
# --------------------------------------------------------------------------- #

def filtrer_datasets(catalogue: pd.DataFrame, mots_cles_theme: list[str]) -> pd.DataFrame:
    """
    Garde les datasets dont title/tags/description contient au moins un
    mot-clé du thème (insensible casse/accents, frontière de mot,
    pluriel toléré : "comptage" matche "comptages").
    """
    mots = sorted({_normaliser(m) for m in mots_cles_theme if m})
    print(f"!! lots '{mots}' ")
    if not mots:
        return catalogue.iloc[0:0]
    regex = re.compile("|".join(r"\b" + re.escape(m) + r"[a-z]{0,2}\b" for m in mots))
    print(f"!! lots '{regex}' ")
    colonnes = catalogue[COLONNES_TEXTE].fillna("")
    texte = colonnes.agg(" ".join, axis=1).apply(_normaliser)
    return catalogue[texte.apply(lambda t: bool(regex.search(t)))].copy()



In [6]:


# --------------------------------------------------------------------------- #
# Étape 3 : scoring embeddings (cosinus + marge)
# --------------------------------------------------------------------------- #

def embedder_datasets(modele, datasets, taille_lot=TAILLE_LOT) -> np.ndarray:
    """Embeddings normalisés des datasets (un passage unique pour tous les thèmes)."""
    textes = [PREFIXE_PASSAGE + texte_dataset(row) for _, row in datasets.iterrows()]
    return modele.encode(
        textes,
        batch_size=taille_lot,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )


def embedder_sous_theme(modele, termes: str) -> np.ndarray:
    """Embedding d'un sous-thème : centroïde renormalisé de ses termes (préfixe query)."""
    phrases = phrases_sous_theme(termes)
    if not phrases:
        raise ValueError("Sous-thème sans termes descriptifs.")
    vecteurs = modele.encode(
        [PREFIXE_QUERY + p for p in phrases],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    centre = vecteurs.mean(axis=0)
    return centre / (np.linalg.norm(centre) + 1e-12)


def scorer_theme(
    modele: SentenceTransformer,
    emb_datasets: np.ndarray,
    sous_themes: dict[str, str],
    seuil_marge: float = SEUIL_MARGE,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame], pd.DataFrame]:
    """
    Pour les datasets d'un thème, calcule cosinus et marge vers chaque
    sous-thème DU THÈME (les concurrents sont les autres sous-thèmes du
    même thème ; les sous-thèmes d'autres thèmes ne sont pas des
    concurrents puisque le dataset a déjà été filtré sur son thème).

    Renvoie (datasets_scores, par_sous_theme, synthese) :
      - datasets_scores : colonnes score__<st> et marge__<st> par sous-thème
      - par_sous_theme   : {sous-thème: datasets retenus (marge >= seuil), triés}
      - synthese         : une ligne par sous-thème (effectifs, stats de marge)
    """
    noms_st = list(sous_themes.keys())
    emb_st = {st: embedder_sous_theme(modele, sous_themes[st]) for st in noms_st}
    matrice = np.column_stack([emb_datasets @ emb_st[st] for st in noms_st])

    scores = pd.DataFrame(index=range(len(emb_datasets)))
    for i, st in enumerate(noms_st):
        autres = np.delete(matrice, i, axis=1)
        marge = (matrice[:, i] - autres.max(axis=1)) if autres.shape[1] else np.nan
        scores[f"score__{st}"] = matrice[:, i]
        scores[f"marge__{st}"] = marge

    par_sous_theme = {}
    lignes_synthese = []
    for st in noms_st:
        col = f"marge__{st}"
        ok = scores[col] >= seuil_marge
        par_sous_theme[st] = scores[ok].sort_values(col, ascending=False)
        lignes_synthese.append({
            "sous_theme": st,
            "n_datasets_theme": len(emb_datasets),
            "n_retenus": int(ok.sum()),
            "marge_max": float(np.nanmax(scores[col])),
            "marge_mediane": float(np.nanmedian(scores[col])),
            "cosinus_max": float(scores[f"score__{st}"].max()),
        })
    synthese = pd.DataFrame(lignes_synthese)
    return scores, par_sous_theme, synthese



In [7]:


# --------------------------------------------------------------------------- #
# Pipeline complet
# --------------------------------------------------------------------------- #

def pipeline_complet(
    chemin_sous_themes=CHEMIN_SOUS_THEMES,
    chemin_catalogue=CHEMIN_CATALOGUE,
    dossier_sortie=DOSSIER_SORTIE,
    nom_modele=MODELE,
    seuil_marge=SEUIL_MARGE,
    themes_a_traiter: list[str] | None = None,  # None = tous
    verbose: bool = True,
) -> dict[str, pd.DataFrame]:
    """
    Exécute la chaîne complète et écrit les fichiers de sortie.

    Retourne {nom de thème: DataFrame des datasets du thème avec scores}
    pour inspection interactive.
    """
    dossier = Path(dossier_sortie)
    dossier.mkdir(parents=True, exist_ok=True)

    # --- Taxonomie et catalogue (chargés une seule fois) ---
    taxo = charger_taxonomie(chemin_sous_themes)
    catalogue = pd.read_csv(chemin_catalogue, sep=";", index_col=0)
    themes = themes_de_la_taxonomie(taxo)
    if themes_a_traiter is not None:
        themes = {t: mc for t, mc in themes.items()
                  if any(_normaliser(t) == _normaliser(k) for k in themes_a_traiter)}
    if verbose:
        print(f"Taxonomie : {len(themes)} thèmes, {len(taxo)} lignes de sous-thèmes")
        print(f"Catalogue : {len(catalogue)} datasets")

    # --- Modèle chargé une seule fois pour tout le pipeline ---
    if verbose:
        print(f"Chargement du modèle {nom_modele} (CPU)...")
    modele = SentenceTransformer(nom_modele, device="cpu")

    tous_resultats: dict[str, pd.DataFrame] = {}
    synthese_globale = []

    for nom_theme, mots_cles_theme in themes.items():
        slug_theme = _normaliser(nom_theme).replace(" ", "-")
        sous_themes = sous_themes_du_theme(taxo, nom_theme)
        if verbose:
            print(f"slug_theme '{slug_theme}' et sous_themes: '{sous_themes}'")
        if not sous_themes:
            if verbose:
                print(f"!! Thème '{nom_theme}' : aucun sous-thème trouvé, ignoré")
            continue

        # 1. Filtre par thème
        datasets_theme = filtrer_datasets(catalogue, mots_cles_theme)
        if verbose:
            print(f"\n=== Thème '{nom_theme}' : {len(datasets_theme)} datasets, "
                  f"{len(sous_themes)} sous-thèmes ===")
        if datasets_theme.empty:
            if verbose:
                print("   Aucun dataset, thème ignoré")
            continue
        datasets_theme = datasets_theme.reset_index(drop=True)

        # Sauvegarde de la sélection par mots-clés (avant classification
        # en sous-thèmes) : utile pour re-traiter les sous-thèmes sans
        # refaire le filtrage ni ré-embedder tout le catalogue.
        datasets_theme.to_csv(
            dossier / f"datasets_filtres__{slug_theme}.csv", sep=";", index=False
        )

        # 2. Embeddings des datasets du thème
        emb = embedder_datasets(modele, datasets_theme)

        # 3. Scores et marges vers les sous-thèmes du thème
        scores, par_st, synthese = scorer_theme(
            modele, emb, sous_themes, seuil_marge=seuil_marge
        )

        # 4. Assemblage et écritures
        resultat = pd.concat([datasets_theme, scores], axis=1)
        tous_resultats[nom_theme] = resultat
        resultat.to_csv(dossier / f"datasets__{slug_theme}.csv", sep=";", index=False)

        for st, df_st in par_st.items():
            slug_st = _normaliser(st).replace(" ", "-").replace("/", "-")
            vue = resultat.loc[df_st.index][
                ["id", "url", "title", "tags", f"score__{st}", f"marge__{st}"]
            ].sort_values(f"marge__{st}", ascending=False)
            vue.to_csv(dossier / f"par_sous_theme__{slug_theme}__{slug_st}.csv",
                       sep=";", index=False)
            if verbose:
                print(f"   {st} : {len(vue)} datasets retenus (marge >= {seuil_marge})")

        synthese.insert(0, "theme", nom_theme)
        synthese_globale.append(synthese)

    # --- Synthèse globale ---
    if synthese_globale:
        pd.concat(synthese_globale, ignore_index=True).to_csv(
            dossier / "synthese.csv", sep=";", index=False
        )
        if verbose:
            print(f"\nSynthèse écrite dans {dossier / 'synthese.csv'}")

    return tous_resultats



In [ ]:

# --------------------------------------------------------------------------- #
# Exemple d'utilisation
# --------------------------------------------------------------------------- #

if __name__ == "__main__":
    # Tout le catalogue, tous les thèmes :
    resultats = pipeline_complet()

    # Ou pour tester sur quelques thèmes d'abord :
    # resultats = pipeline_complet(themes_a_traiter=["Stationnement", "Vélo"])

    # Ou avec le modèle plus léger :
    # resultats = pipeline_complet(nom_modele="intfloat/multilingual-e5-base")

In [ ]:
resultats = pipeline_complet(chemin_sous_themes=CHEMIN_SOUS_THEMES, chemin_catalogue=CHEMIN_CATALOGUE, seuil_marge=0.015, verbose=True)

Taxonomie : 14 thèmes, 39 lignes de sous-thèmes
Catalogue : 74633 datasets
Chargement du modèle intfloat/multilingual-e5-large (CPU)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2625.97it/s]


slug_theme 'transport-collectif' et sous_themes: '{'Offre planifiée': 'GTFS, NETEX, offre planifiée de transport public, offre de Transport collectif, transport public, transports collectifs, transport en commun, bus, train, tram,arrêts, réseau, ligne, itinéraires', 'Offre temps réel / perturbations': 'GTFS-RT, SIRI, horaires temps réel, horaires de passage, perturbations transport collectif', 'Tarification': 'tarifs, tarification, prix ticket, titres de transport, abonnement', 'Accessibilité': 'accessiblité transport, accessibilité PMR, accessibilité aux arrêts', 'Fréquentation et usage': 'fréquentation, remplissage transport, saturation transport, validations, validation billettique, flux passagers, flux voyageurs', 'Qualité de service et performance': 'ponctualité, performance réseau transport, retards, niveau de service'}'
!! lots '['bus', 'train', 'tram', 'transport collectif', 'transport en commun', 'transport public', 'transports collectifs']' 
!! lots 're.compile('\\bbus[a-z]{0

/tmp/ipykernel_751/1546205460.py:69: RuntimeWarning: All-NaN axis encountered
  "marge_max": float(np.nanmax(scores[col])),
/tmp/ipykernel_751/1546205460.py:70: RuntimeWarning: All-NaN slice encountered
  "marge_mediane": float(np.nanmedian(scores[col])),


   Aires de covoiturage : 0 datasets retenus (marge >= 0.015)
slug_theme 'marche' et sous_themes: '{'Réseau piéton': 'réseau piéton, voirie piétonnes, chemins piétons, sentiers, cheminement', 'Comptages piétons': 'comptage piéton, fréquentation randonneurs, comptage piétons'}'
!! lots '['marche', 'pieton']' 
!! lots 're.compile('\\bmarche[a-z]{0,2}\\b|\\bpieton[a-z]{0,2}\\b')' 

=== Thème 'Marche' : 4451 datasets, 2 sous-thèmes ===


In [9]:
resultats = pipeline_complet(chemin_sous_themes=CHEMIN_SOUS_THEMES, chemin_catalogue=CHEMIN_CATALOGUE, seuil_marge=0.15, themes_a_traiter=["Véhicules", "Vélo"], verbose=True)

Taxonomie : 2 thèmes, 39 lignes de sous-thèmes
Catalogue : 23775 datasets
Chargement du modèle intfloat/multilingual-e5-large (CPU)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3204.39it/s]


slug_theme 'velo' et sous_themes: '{'Aménagements cyclables': 'aménagements cyclables, pistes cyclables, voies vertes, bande cyclable', 'Stationnement vélo': 'stationnement vélo, arceaux, parc à vélo, box vélos', 'Comptages et fréquentation cyclistes': 'comptages vélo, compteurs vélo, fréquentation cyclistes', 'Enquêtes et pratiques cyclables': 'enquête vélo, pratiques usagers vélo, enquête cyclistes', 'Usage des vélos en libre service': 'usage VLS, fréquentation vélo libre service'}'
!! lots '['velo']' 
!! lots 're.compile('\\bvelo[a-z]{0,2}\\b')' 

=== Thème 'Vélo' : 974 datasets, 5 sous-thèmes ===
   Aménagements cyclables : 0 datasets retenus (marge >= 0.15)
   Stationnement vélo : 0 datasets retenus (marge >= 0.15)
   Comptages et fréquentation cyclistes : 0 datasets retenus (marge >= 0.15)
   Enquêtes et pratiques cyclables : 0 datasets retenus (marge >= 0.15)
   Usage des vélos en libre service : 0 datasets retenus (marge >= 0.15)
slug_theme 'vehicules' et sous_themes: '{'Parc d